In [ ]:
from ultralytics import YOLO
import cv2

# Load model
model = YOLO('yolov8n.pt')  # or yolov8s.pt for better accuracy

# Open webcam
cap = cv2.VideoCapture(0)

print("Tracking started. Press 'q' to quit.")
print("Person IDs will persist when they leave and return to frame.")

# Store color for each ID for consistent coloring
id_colors = {}

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Run tracking (track=True, persist=True for maintaining IDs)
    results = model.track(
        frame,
        persist=True,      # Keep tracking even if object disappears
        conf=0.5,          # Confidence threshold
        classes=[0],       # Only track people (class 0)
        verbose=False      # Don't print progress
    )[0]
    
    # Get tracked objects
    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        track_ids = results.boxes.id.cpu().numpy().astype(int)
        confidences = results.boxes.conf.cpu().numpy()
        
        for box, track_id, confidence in zip(boxes, track_ids, confidences):
            x1, y1, x2, y2 = map(int, box)
            
            # Assign a consistent color to each ID
            if track_id not in id_colors:
                # Generate a color based on ID (for consistency)
                color_id = track_id * 100 % 255
                id_colors[track_id] = (
                    (color_id * 7) % 256,
                    (color_id * 13) % 256,
                    (color_id * 17) % 256
                )
            
            color = id_colors[track_id]
            
            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            # Draw ID and confidence
            label = f"ID {track_id}: {confidence:.1%}"
            cv2.putText(frame, label, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Mark the center point
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            cv2.circle(frame, (center_x, center_y), 4, color, -1)
    
    # Show frame
    cv2.imshow('Person Tracking', frame)
    
    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

2026-02-06 10:59:05.550 python[26709:1717713] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


Tracking started. Press 'q' to quit.
Person IDs will persist when they leave and return to frame.


error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'rectangle'
> Overload resolution failed:
>  - Scalar value for argument 'color' is not numeric
>  - Can't parse 'rec'. Expected sequence length 4, got 2
>  - Scalar value for argument 'color' is not numeric
>  - Can't parse 'rec'. Expected sequence length 4, got 2


: 

In [1]:
from ultralytics import YOLO
import cv2

# Load model
model = YOLO('yolov8n.pt')

# Predefined colors for up to 10 different IDs
COLORS = [
    (255, 0, 0),    # Blue
    (0, 255, 0),    # Green
    (0, 0, 255),    # Red
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 255, 255),  # Yellow
    (128, 0, 0),    # Dark Blue
    (0, 128, 0),    # Dark Green
    (0, 0, 128),    # Dark Red
    (128, 128, 0)   # Dark Cyan
]

cap = cv2.VideoCapture(0)

print("Simple Person Tracking")
print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Run tracking
    results = model.track(
        frame,
        persist=True,
        conf=0.5,
        classes=[0],
        verbose=False
    )[0]
    
    # Draw detections
    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy().astype(int)
        ids = results.boxes.id.cpu().numpy().astype(int)
        
        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = box
            
            # Get color based on track ID (cycle through COLORS)
            color_idx = (track_id - 1) % len(COLORS)
            color = COLORS[color_idx]
            
            # Draw rectangle (ensure color is tuple of ints)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            # Draw ID label
            label = f"ID {track_id}"
            cv2.putText(frame, label, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    cv2.imshow('Person Tracking', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

2026-02-06 11:00:40.420 python[26731:1719420] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


Simple Person Tracking
Press 'q' to quit


In [2]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8n.pt')
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model.track(frame, persist=True, conf=0.5, classes=[0], verbose=False)[0]
    
    if results.boxes is not None:
        for box in results.boxes:
            # Get coordinates as integers
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Get track ID if available
            if box.id is not None:
                track_id = int(box.id)
                label = f"ID {track_id}"
                color = (0, 255, 0)  # Green
            else:
                label = "Person"
                color = (255, 0, 0)  # Blue
            
            # Draw with proper color tuple
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    cv2.imshow('Tracking', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8n.pt')
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model.track(frame, persist=True, conf=0.8, classes=[0], verbose=False)[0]
    
    if results.boxes is not None:
        for box in results.boxes:
            # Get coordinates
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Calculate center and dimensions
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            width = x2 - x1
            height = y2 - y1
            
            # Inner rectangle (half the width and height = 4x smaller area)
            inner_width = width // 4
            inner_height = height // 2
            inner_x1 = center_x - inner_width // 2
            inner_y1 = center_y - inner_height // 2
            inner_x2 = center_x + inner_width // 2
            inner_y2 = center_y + inner_height // 2
            
            # Get ID and colors
            if box.id is not None:
                track_id = int(box.id)
                label = f"ID {track_id}"
                color = (0, 255, 0)  # Green
            else:
                label = "Person"
                color = (255, 0, 0)  # Blue
            
            # Draw rectangles
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)  # Outer
            cv2.rectangle(frame, (inner_x1, inner_y1), (inner_x2, inner_y2), (255, 255, 0), 2)  # Inner (yellow)
            
            # Draw label
            cv2.putText(frame, label, (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    cv2.imshow('Dual Rectangle Tracking', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

: 

In [16]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8n.pt')
cap = cv2.VideoCapture(0)

# Get frame dimensions to calculate center point
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Calculate center point of the screen
screen_center_x = frame_width // 2
screen_center_y = frame_height // 2

print(f"Screen center point: ({screen_center_x}, {screen_center_y})")
print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Draw the center point on screen (red cross)
    point_size = 10
    cv2.line(frame, (screen_center_x - point_size, screen_center_y), 
             (screen_center_x + point_size, screen_center_y), (0, 0, 255), 3)
    cv2.line(frame, (screen_center_x, screen_center_y - point_size), 
             (screen_center_x, screen_center_y + point_size), (0, 0, 255), 3)
    
    # Run tracking
    results = model.track(frame, persist=True, conf=0.5, classes=[0], verbose=False)[0]
    
    if results.boxes is not None:
        for box in results.boxes:
            # Get coordinates
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Calculate center and dimensions
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            width = x2 - x1
            height = y2 - y1
            
            # Inner rectangle (half the width and height = 4x smaller area)
            inner_width = width // 2
            inner_height = height // 2
            inner_x1 = center_x - inner_width // 2
            inner_y1 = center_y - inner_height // 2
            inner_x2 = center_x + inner_width // 2
            inner_y2 = center_y + inner_height // 2
            
            # Get ID and colors
            if box.id is not None:
                track_id = int(box.id)
                label = f"ID {track_id}"
                color = (0, 255, 0)  # Green
            else:
                label = "Person"
                color = (255, 0, 0)  # Blue
            
            # Check if the inner rectangle contains the screen center point
            center_point_in_inner_rect = (
                screen_center_x >= inner_x1 and 
                screen_center_x <= inner_x2 and 
                screen_center_y >= inner_y1 and 
                screen_center_y <= inner_y2
            )
            
            # Visual indicator for the check
            if center_point_in_inner_rect:
                # Highlight the center point when it's inside the inner rectangle
                cv2.circle(frame, (screen_center_x, screen_center_y), 15, (0, 255, 255), 3)
                
                # Change inner rectangle color to indicate containment
                inner_rect_color = (0, 165, 255)  # Orange
                status_text = "CENTER INSIDE INNER RECT"
                status_color = (0, 165, 255)
            else:
                inner_rect_color = (255, 255, 0)  # Yellow
                status_text = "center outside"
                status_color = (100, 100, 100)
            
            # Draw rectangles
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)  # Outer
            cv2.rectangle(frame, (inner_x1, inner_y1), (inner_x2, inner_y2), inner_rect_color, 2)  # Inner
            
            # Draw label with status
            full_label = f"{label} - {status_text}"
            cv2.putText(frame, full_label, (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2)
            
            # Draw a line from person center to screen center
            cv2.line(frame, (center_x, center_y), (screen_center_x, screen_center_y), 
                    (200, 200, 200), 1)
            
            # Display the boolean value
            bool_text = f"Center in inner rect: {center_point_in_inner_rect}"
            cv2.putText(frame, bool_text, (x1, y2+20), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # Display screen center coordinates
    center_info = f"Screen Center: ({screen_center_x}, {screen_center_y})"
    cv2.putText(frame, center_info, (10, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # Display frame dimensions
    dim_info = f"Frame: {frame_width}x{frame_height}"
    cv2.putText(frame, dim_info, (10, 60), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    
    cv2.imshow('Dual Rectangle Tracking with Center Point Check', frame)
    
    # Optional: Print to console when center point enters/exits inner rectangle
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('d'):  # Press 'd' to debug/toggle additional info
        # You can add debug functionality here
        pass

cap.release()
cv2.destroyAllWindows()

Screen center point: (960, 540)
Press 'q' to quit


KeyboardInterrupt: 

In [2]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8s.pt')
cap = cv2.VideoCapture(0)

# Get screen dimensions
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
screen_center_x = frame_width // 2
screen_center_y = frame_height // 2

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Draw center point
    cv2.circle(frame, (screen_center_x, screen_center_y), 6, (0, 0, 255), -1)
    
    results = model.track(frame, persist=True, conf=0.7, classes=[0], verbose=False)[0]
    
    if results.boxes is not None:
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Inner rectangle (4x smaller)
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            inner_width = (x2 - x1) // 4
            inner_height = (y2 - y1) // 2
            inner_x1 = center_x - inner_width // 2
            inner_y1 = center_y - inner_height // 2
            inner_x2 = center_x + inner_width // 2
            inner_y2 = center_y + inner_height // 2
            
            # Check if center point is inside inner rectangle
            center_in_inner = (
                screen_center_x >= inner_x1 and 
                screen_center_x <= inner_x2 and 
                screen_center_y >= inner_y1 and 
                screen_center_y <= inner_y2
            )
            
            # Colors based on the check
            outer_color = (0, 255, 0) if center_in_inner else (255, 0, 0)
            inner_color = (0, 255, 255) if center_in_inner else (255, 255, 0)
            
            # Draw rectangles
            cv2.rectangle(frame, (x1, y1), (x2, y2), outer_color, 2)
            cv2.rectangle(frame, (inner_x1, inner_y1), (inner_x2, inner_y2), inner_color, 2)
            
            # Get ID
            track_id = int(box.id) if box.id is not None else "?"
            
            # Display info
            status = "INSIDE" if center_in_inner else "OUTSIDE"
            label = f"ID {track_id}: Center {status}"
            cv2.putText(frame, label, (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, outer_color, 2)
    
    cv2.imshow('Center Point Check', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [1]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8s.pt')
cap = cv2.VideoCapture(0)
Status_screen_past=[]
Status_screen=[]
# Get screen dimensions
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
screen_center_x = frame_width // 2
screen_center_y = frame_height // 2

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Draw center point
    cv2.circle(frame, (screen_center_x, screen_center_y), 6, (0, 0, 255), -1)
    
    results = model.track(frame, persist=True, conf=0.6, classes=[0], verbose=False)[0]
    
    if results.boxes is not None:
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Inner rectangle (4x smaller)
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            inner_width = (x2 - x1) // 4
            inner_height = (y2 - y1) // 2
            inner_x1 = center_x - inner_width // 2
            inner_y1 = center_y - inner_height // 2
            inner_x2 = center_x + inner_width // 2
            inner_y2 = center_y + inner_height // 2
            
            # Check if center point is inside inner rectangle
            center_in_inner = (
                screen_center_x >= inner_x1 and 
                screen_center_x <= inner_x2 and 
                screen_center_y >= inner_y1 and 
                screen_center_y <= inner_y2
            )
            
            # Colors based on the check
            
            # Draw rectangles
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.rectangle(frame, (inner_x1, inner_y1), (inner_x2, inner_y2), (0, 255, 200), 2)
            
            # Get ID
            track_id = int(box.id) if box.id is not None else "?"
            

            
            # Display info
            status = "True" if center_in_inner else "False"
            
            if status == "True":
                left_right="inside"
            elif inner_x1> screen_center_x:
                left_right="left"
            else:
                left_right="right"
            
            
            
            label = f"ID {track_id}: Center {status} and is {left_right}"
            if box.id != None:
                if int(box.id)> len(Status_screen):
                    Status_screen.append(eval(status))
                else:
                    Status_screen[int(box.id)-1]=eval(status)
                print(Status_screen)
                if Status_screen != Status_screen_past:
                    print(Status_screen)
                    Status_screen_past=Status_screen
            
            cv2.putText(frame, label, (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 200), 2)
    
    cv2.imshow('Center Point Check', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


2026-04-30 21:20:46.492 python[61978:29582787] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


WARNING ⚠️ not enough matching points
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[True]
[False]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[False]
[True]
[True]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[True]
[True]
[True]
[True]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[True]
[True]
[True]
[True]
[True]
[False]
[False]
[False]
[False]
[False]
[False]
[False]
[True]
[True]
[True]
[True]
[True]
[True]
[True]
[False]
[False]
[False]
[False]
[False]
[Fal

KeyboardInterrupt: 

In [3]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8n.pt')
cap = cv2.VideoCapture(0)

# Get screen dimensions
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
screen_center_x = frame_width // 2
screen_center_y = frame_height // 2

print(f"Screen center: ({screen_center_x}, {screen_center_y})")
print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Draw center point (red cross)
    point_size = 8
    cv2.line(frame, (screen_center_x - point_size, screen_center_y), 
             (screen_center_x + point_size, screen_center_y), (0, 0, 255), 2)
    cv2.line(frame, (screen_center_x, screen_center_y - point_size), 
             (screen_center_x, screen_center_y + point_size), (0, 0, 255), 2)
    
    results = model.track(frame, persist=True, conf=0.5, classes=[0], verbose=False)[0]
    
    if results.boxes is not None:
        for box in results.boxes:
            # Get coordinates
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Calculate center and dimensions
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            width = x2 - x1
            height = y2 - y1
            
            # Inner rectangle (half the width and height = 4x smaller area)
            inner_width = width // 2
            inner_height = height // 2
            inner_x1 = center_x - inner_width // 2
            inner_y1 = center_y - inner_height // 2
            inner_x2 = center_x + inner_width // 2
            inner_y2 = center_y + inner_height // 2
            
            # Get ID and colors
            if box.id is not None:
                track_id = int(box.id)
                label = f"ID {track_id}"
                color = (0, 255, 0)  # Green
            else:
                label = "Person"
                color = (255, 0, 0)  # Blue
            
            # NEW: Check if center point is LEFT or RIGHT of the rectangle
            is_left_of_rect = screen_center_x < x1  # True if center is left of rectangle's left edge
            is_right_of_rect = screen_center_x > x2  # True if center is right of rectangle's right edge
            
            # Draw rectangles (unchanged)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)  # Outer
            cv2.rectangle(frame, (inner_x1, inner_y1), (inner_x2, inner_y2), (255, 255, 0), 2)  # Inner (yellow)
            
            # Draw label (unchanged)
            cv2.putText(frame, label, (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # NEW: Display LEFT/RIGHT information
            position_text = ""
            position_color = (200, 200, 200)  # Gray
            
            if is_left_of_rect:
                position_text = "Center is LEFT of person"
                position_color = (0, 100, 255)  # Orange
            elif is_right_of_rect:
                position_text = "Center is RIGHT of person"
                position_color = (255, 100, 0)  # Blue-ish
            else:
                position_text = "Center is HORIZONTALLY WITHIN person"
                position_color = (0, 255, 0)  # Green
            
            # Display position info
            cv2.putText(frame, position_text, (x1, y2 + 20), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, position_color, 1)
            
            # NEW: Display the boolean values
            bool_text = f"Left: {is_left_of_rect}, Right: {is_right_of_rect}"
            cv2.putText(frame, bool_text, (x1, y2 + 40), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            # NEW: Draw visual indicators
            if is_left_of_rect:
                # Draw arrow from center pointing left
                cv2.arrowedLine(frame, (screen_center_x, screen_center_y),
                               (screen_center_x - 50, screen_center_y), 
                               (0, 100, 255), 2, tipLength=0.3)
            elif is_right_of_rect:
                # Draw arrow from center pointing right
                cv2.arrowedLine(frame, (screen_center_x, screen_center_y),
                               (screen_center_x + 50, screen_center_y), 
                               (255, 100, 0), 2, tipLength=0.3)
    
    # Display frame info (unchanged)
    cv2.putText(frame, f"Screen Center: ({screen_center_x}, {screen_center_y})", 
               (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    cv2.imshow('Dual Rectangle Tracking with Left/Right Detection', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Screen center: (960, 540)
Press 'q' to quit


KeyboardInterrupt: 